# 03 - Custom Vegetation Index Builder

**What:** Let the user type an arbitrary band expression (e.g. `(B8 - B4) / (B8 + B4)`), name it, and compute it on a Sentinel-2 image via `ee.Image.expression`.

**Why:** The RAVI plugin ships with ~20 built-in indices, but agronomists frequently need a one-off formula. Rather than hard-coding every variant, we expose a free-form expression that maps the 12 Sentinel-2 bands into the GEE expression engine. The expression and its name are persisted in `QSettings` (`ravi_plugin/custom_expression`, `ravi_plugin/custom_expression_name`) so they survive across sessions and get added to every index dropdown as `"<name> (custom)"`.

**Legacy reference:**
- `legacy:ravi_dialog.py` -> `setup_custom_clicked` / `add_custom_index_clicked` (~L704-750): dialog wiring + QSettings persistence.
- `legacy:ravi_dialog.py` -> the `custom(image)` branch inside `calculate_vegetation_index` (~L2250-2270): the exact expression evaluation and band dictionary reproduced below.
- `legacy:ui/custom_index.ui`: the band reference table and example expressions.

**Allowed bands:** B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12 — each scaled `/10000` (S2 reflectance scale factor).

In [ ]:
# --- Setup ---
import os
import ee
import pandas as pd

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)
print("Earth Engine initialized.")

In [ ]:
# --- AOI (from project shapefile, same area as dev/testing.ipynb) + dates + sample image ---
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


aoi = load_aoi_from_shapefile("contorno_area_total.zip").geometry()

START_DATE = "2023-06-01"
END_DATE = "2023-06-30"

# Median composite over the period, low cloud cover. The Harmonized SR collection
# exposes the bands B1..B12 plus B8A that the custom builder relies on.
collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

img = collection.median().clip(aoi)

print("Image count in collection:", collection.size().getInfo())
print("Bands available:", img.bandNames().getInfo())

In [ ]:
# --- User-defined custom index (this is what the dialog captures + persists to QSettings) ---
# In the plugin: self.custom_expression / self.custom_expression_name,
# stored under ravi_plugin/custom_expression and ravi_plugin/custom_expression_name.
CUSTOM_EXPRESSION = "(B8 - B4) / (B8 + B4)"
CUSTOM_NAME = "MyNDVI"

# In the plugin the dropdown entry becomes "<name> (custom)":
custom_index_name = CUSTOM_NAME + " (custom)"
print("Expression:", CUSTOM_EXPRESSION)
print("Dropdown label:", custom_index_name)

In [ ]:
# --- Compute the custom index (exact port of the legacy `custom(image)` branch) ---
# Source: legacy:ravi_dialog.py, calculate_vegetation_index -> custom(image).
# Every allowed band is selected and divided by 10000, then fed to image.expression
# under its band name. The result is renamed (legacy renames to "index"; here we use
# CUSTOM_NAME so the validation cell can compare cleanly).
def custom(image, expression):
    # Add all bands to the custom index calculation
    band1 = image.select("B1").divide(10000)   # Coastal aerosol
    band2 = image.select("B2").divide(10000)   # Blue
    band3 = image.select("B3").divide(10000)   # Green
    band4 = image.select("B4").divide(10000)   # Red
    band5 = image.select("B5").divide(10000)   # Red Edge 1
    band6 = image.select("B6").divide(10000)   # Red Edge 2
    band7 = image.select("B7").divide(10000)   # Red Edge 3
    band8 = image.select("B8").divide(10000)   # NIR
    band8a = image.select("B8A").divide(10000) # Narrow NIR
    band9 = image.select("B9").divide(10000)   # Water vapor
    band11 = image.select("B11").divide(10000) # SWIR 1
    band12 = image.select("B12").divide(10000) # SWIR 2

    return image.expression(
        expression,
        {
            "B1": band1,
            "B2": band2,
            "B3": band3,
            "B4": band4,
            "B5": band5,
            "B6": band6,
            "B7": band7,
            "B8": band8,
            "B8A": band8a,
            "B9": band9,
            "B11": band11,
            "B12": band12,
        },
    )

custom_index = custom(img, CUSTOM_EXPRESSION).rename(CUSTOM_NAME)
print("Custom index band:", custom_index.bandNames().getInfo())

In [ ]:
# --- Validation: mean over AOI, and sanity check against built-in NDVI ---
# Built-in NDVI in the plugin is: image.normalizedDifference(["B8", "B4"]).
# normalizedDifference computes (B8 - B4)/(B8 + B4) on the RAW band values; the
# common /10000 scale factor cancels in the ratio, so the custom NDVI expression
# (which divides each band by 10000) must yield the identical result.
builtin_ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI_builtin")

SCALE = 10  # native resolution of B4/B8 (10 m)

custom_mean = custom_index.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=SCALE, maxPixels=1e9
).get(CUSTOM_NAME).getInfo()

builtin_mean = builtin_ndvi.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=SCALE, maxPixels=1e9
).get("NDVI_builtin").getInfo()

results = pd.DataFrame(
    [
        {"index": custom_index_name, "expression": CUSTOM_EXPRESSION, "mean": custom_mean},
        {"index": "NDVI (built-in)", "expression": "normalizedDifference([B8,B4])", "mean": builtin_mean},
    ]
)
print(results.to_string(index=False))

diff = abs(custom_mean - builtin_mean)
print("\nAbsolute difference:", diff)
assert diff < 1e-9, "Custom NDVI expression should match built-in NDVI (scale factor cancels)"
print("Sanity check passed: custom expression equals built-in NDVI.")

## Sentinel-2 band reference

Allowed band names for the custom expression (from `legacy:ui/custom_index.ui`). Each is divided by 10000 before evaluation.

| Band | Description       | Detail                   | Resolution |
|------|-------------------|--------------------------|------------|
| B1   | Coastal aerosol   | Coastal aerosol          | 60 m       |
| B2   | Blue              | Blue                     | 10 m       |
| B3   | Green             | Green                    | 10 m       |
| B4   | Red               | Red                      | 10 m       |
| B5   | Red Edge 1        | Red Edge 1               | 20 m       |
| B6   | Red Edge 2        | Red Edge 2               | 20 m       |
| B7   | Red Edge 3        | Red Edge 3               | 20 m       |
| B8   | NIR               | Near Infrared            | 10 m       |
| B8A  | Narrow NIR        | Narrow Near Infrared     | 20 m       |
| B9   | Water vapor       | Water vapor              | 60 m       |
| B11  | SWIR 1            | Shortwave Infrared 1     | 20 m       |
| B12  | SWIR 2            | Shortwave Infrared 2     | 20 m       |

**Example expressions (from the UI help panel):**
- NDVI: `(B8 - B4) / (B8 + B4)`
- EVI: `2.5 * ((B8 - B4) / (B8 + 6 * B4 - 7.5 * B2 + 1))`
- SAVI: `(1 + 0.5) * ((B8 - B4) / (B8 + B4 + 0.5))`
- NDRE: `(B8 - B5) / (B8 + B5)`
- NDMI: `(B8 - B11) / (B8 + B11)`
- Conditional NDVI: `(B4 > 0.2) ? ((B8 - B4) / (B8 + B4)) : 0`
- Default (Average): `((B1 + B2 + B3 + B4 + B5 + B6 + B7 + B8 + B8A + B9 + B11 + B12) / 12)`